# ThermalSense — XGBoost Training
Loads historical OGN data (IGC or CSV), labels thermals, trains, evaluates, saves.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'backend'))

import numpy as np
import pandas as pd
import xgboost as xgb
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder

## 1. Load & parse IGC / CSV flight logs

In [ ]:
def parse_igc(path: str) -> pd.DataFrame:
    """Parse a single IGC file into a DataFrame of fix records."""
    rows = []
    with open(path) as f:
        for line in f:
            if not line.startswith('B'):
                continue
            # B HHMMSS DDMMmmmN DDDMMmmmE V PPPPP GGGGG
            try:
                hh, mm, ss = int(line[1:3]), int(line[3:5]), int(line[5:7])
                lat_d, lat_m = int(line[7:9]), int(line[9:14]) / 1000
                lat = lat_d + lat_m / 60
                if line[14] == 'S': lat = -lat
                lon_d, lon_m = int(line[15:18]), int(line[18:23]) / 1000
                lon = lon_d + lon_m / 60
                if line[23] == 'W': lon = -lon
                press_alt = int(line[25:30])
                gps_alt   = int(line[30:35])
                rows.append({'hh': hh, 'mm': mm, 'ss': ss,
                             'lat': lat, 'lon': lon,
                             'press_alt': press_alt, 'gps_alt': gps_alt})
            except (ValueError, IndexError):
                continue
    return pd.DataFrame(rows)

def load_dataset(data_dir: str) -> pd.DataFrame:
    """Load all .igc and .csv files from data_dir."""
    frames = []
    p = Path(data_dir)
    for igc in p.glob('**/*.igc'):
        df = parse_igc(str(igc))
        df['source'] = igc.stem
        frames.append(df)
    for csv in p.glob('**/*.csv'):
        df = pd.read_csv(csv)
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f'No IGC or CSV files found in {data_dir}')
    return pd.concat(frames, ignore_index=True)

DATA_DIR = '../data/flights'   # ← point at your flight logs
# df = load_dataset(DATA_DIR)
# Synthetic demo data so notebook runs without real flights:
np.random.seed(42)
n = 5000
df = pd.DataFrame({
    'lat': np.random.uniform(50.5, 52.5, n),
    'lon': np.random.uniform(-1.5, 2.0, n),
    'gps_alt': np.random.randint(100, 2500, n),
    'vario': np.random.normal(0.2, 1.8, n),
    'turn_rate': np.random.normal(0, 12, n),
    'cape': np.random.uniform(0, 1800, n),
    'cin': np.random.uniform(-200, 0, n),
    'solar_ghi': np.random.uniform(0, 900, n),
    'temp_2m': np.random.uniform(10, 28, n),
    'humidity': np.random.uniform(0.3, 0.95, n),
    'wind_speed': np.random.uniform(0, 20, n),
    'wind_dir': np.random.uniform(0, 360, n),
    'lapse_rate': np.random.uniform(4, 9, n),
    'elevation': np.random.uniform(0, 400, n),
    'slope': np.random.uniform(0, 30, n),
    'aspect': np.random.uniform(0, 360, n),
    'hour': np.random.randint(9, 18, n),
    'doy': np.random.randint(1, 366, n),
})
print(df.shape)

## 2. Label thermals
A cell is a thermal if the glider was circling (|turn_rate| > 8 °/s) AND climbing (vario > 1.5 m/s).

In [ ]:
df['circling'] = df['turn_rate'].abs() > 8
df['thermal']  = ((df['vario'] > 1.5) & df['circling']).astype(int)
print(df['thermal'].value_counts())

## 3. Feature engineering

In [ ]:
df['wind_u'] = -df['wind_speed'] * np.sin(np.radians(df['wind_dir']))
df['wind_v'] = -df['wind_speed'] * np.cos(np.radians(df['wind_dir']))
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['doy_sin']  = np.sin(2 * np.pi * df['doy'] / 365)
df['doy_cos']  = np.cos(2 * np.pi * df['doy'] / 365)
df['land_use_heat']   = 0.4   # default CORINE land-use heat factor
df['land_use_albedo'] = 0.20  # default land-use albedo

FEATURE_COLS = [
    'lat', 'lon', 'elevation', 'slope', 'aspect',
    'temp_2m', 'humidity', 'wind_u', 'wind_v',
    'cape', 'cin', 'solar_ghi', 'lapse_rate',
    'land_use_heat', 'land_use_albedo',
    'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos',
]

X = df[FEATURE_COLS].values
y = df['thermal'].values
print('Features:', X.shape, '| Positives:', y.sum())

## 4. Train with cross-validation

In [ ]:
clf = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y == 0).sum() / max(1, y.sum()),
    eval_metric='auc',
    random_state=42,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=cv, scoring='roc_auc')
print(f'CV AUC: {scores.mean():.3f} ± {scores.std():.3f}')

## 5. Final fit & evaluation

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)

y_prob = clf.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.5).astype(int)
print(f'Test AUC: {roc_auc_score(y_test, y_prob):.3f}')
print(classification_report(y_test, y_pred, target_names=['no-thermal', 'thermal']))

## 6. Feature importance

In [ ]:
import matplotlib.pyplot as plt

importances = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
importances.plot.barh(figsize=(8, 6))
plt.title('Feature Importances')
plt.tight_layout()
plt.show()

## 7. Save model

In [ ]:
out = Path('../backend/models/thermal_xgb.json')
out.parent.mkdir(parents=True, exist_ok=True)
clf.save_model(str(out))
print(f'Saved to {out.resolve()}')